In [2]:
from pathlib import Path

VIDEO_DIR = Path("videos")

files = sorted(p for p in VIDEO_DIR.iterdir() if p.is_file())
print(f"{len(files)} files\n")
for p in files:
    print(f"{p.stat().st_size / 1e6:7.1f} MB  {p.name}")

47 files

    0.0 MB  .DS_Store
    4.9 MB  De_problemen_met_arbeidsmigranten_in_Haagse_wijken00003230.mp4
    6.3 MB  De_problemen_met_arbeidsmigranten_in_Haagse_wijken00006013.mp4
    7.7 MB  De_problemen_met_arbeidsmigranten_in_Haagse_wijken00006719.mp4
    8.9 MB  De_problemen_met_arbeidsmigranten_in_Haagse_wijken00009506.mp4
    6.0 MB  De_problemen_met_arbeidsmigranten_in_Haagse_wijken00010452.mp4
    6.1 MB  Desperate_Journey_Shocking_Video_Shows_Risks_Refugee_Families_Take_to_Reach_Europe_Safely_-_Democracy_Now_480p_h264_online-video-cutter.com_00003940.mp4
    9.3 MB  Desperate_Journey_Shocking_Video_Shows_Risks_Refugee_Families_Take_to_Reach_Europe_Safely_-_Democracy_Now_480p_h264_online-video-cutter.com_00005213.mp4
    4.1 MB  Desperate_Journey_Shocking_Video_Shows_Risks_Refugee_Families_Take_to_Reach_Europe_Safely_-_Democracy_Now_480p_h264_online-video-cutter.com_00009300.mp4
    8.0 MB  Desperate_Journey_Shocking_Video_Shows_Risks_Refugee_Families_Take_to_Reach_Europe_Saf

In [3]:
import json, urllib.parse, urllib.request

VIDEO_IDS = ["xtm3LXngnSw", "Q_50066GvYk", "jvWlNsnM7eE", "ONhbxpJp89I", "PGE_l0G5Xcg",
             "CYhe0bn5mnQ", "u8cg5hhHJlA", "iCpFVBuFO1w", "6jswgSJq8Mk", "YM9G3LDayBE",
             "V6MliN1GAqM", "K3r4JxQvVQk"]

for vid in VIDEO_IDS:
    url = f"https://www.youtube.com/watch?v={vid}"
    api = "https://www.youtube.com/oembed?format=json&url=" + urllib.parse.quote(url, safe="")
    info = json.load(urllib.request.urlopen(api))
    print(f"{url} | {info['author_name']} | {info['title']}")

https://www.youtube.com/watch?v=xtm3LXngnSw | Immigratie- en Naturalisatiedienst (IND) | Wie doet wat in de vreemdelingenketen? Een film van IND, COA en DT&V
https://www.youtube.com/watch?v=Q_50066GvYk | Nieuwsuur | De problemen met arbeidsmigranten in Haagse wijken
https://www.youtube.com/watch?v=jvWlNsnM7eE | BBC News | Moment Calais migrants jumped onboard lorries - BBC News
https://www.youtube.com/watch?v=ONhbxpJp89I | NOS op 3 | Waar kunnen de asielzoekers heen?
https://www.youtube.com/watch?v=PGE_l0G5Xcg | Democracy Now! | Desperate Journey: Shocking Video Shows Risks Refugee Families Take to Reach Europe Safely
https://www.youtube.com/watch?v=CYhe0bn5mnQ | NOS op 3 | Zo dodelijk zijn Europa's grenzen
https://www.youtube.com/watch?v=u8cg5hhHJlA | Sky News | Europe's Migration Tragedy: Life and death in the Mediterranean
https://www.youtube.com/watch?v=iCpFVBuFO1w | FRANCE 24 English | In the Netherlands, asylum seekers are no longer welcome • FRANCE 24 English
https://www.youtube

In [6]:
# One-off: rename clips to <source>_<nn>.mp4 and write videos.csv (one row per clip)
from pathlib import Path
from collections import Counter
import pandas as pd

VIDEO_DIR = Path("videos")
CSV_PATH  = Path("videos.csv")
DRY_RUN   = False      # True = only print the plan; False = actually rename and write the CSV

# filename prefix: (source_id, source_title, outlet, source_url)
SOURCES = {
    "De_problemen_met_arbeidsmigranten": ("nieuwsuur_denhaag", "De problemen met arbeidsmigranten in Haagse wijken", "Nieuwsuur", "https://www.youtube.com/watch?v=Q_50066GvYk"),
    "Desperate_Journey":  ("democracynow_journey", "Desperate Journey: Shocking Video Shows Risks Refugee Families Take to Reach Europe Safely", "Democracy Now!", "https://www.youtube.com/watch?v=PGE_l0G5Xcg"),
    "Europes_Migration":  ("sky_mediterranean", "Europe's Migration Tragedy: Life and death in the Mediterranean", "Sky News", "https://www.youtube.com/watch?v=u8cg5hhHJlA"),
    "Greek_police":       ("france24_lesbos", "Greek police begin moving asylum seekers into new Lesbos camp", "FRANCE 24 English", "https://www.youtube.com/watch?v=6jswgSJq8Mk"),
    "In_the_Netherlands": ("france24_nl", "In the Netherlands, asylum seekers are no longer welcome", "FRANCE 24 English", "https://www.youtube.com/watch?v=iCpFVBuFO1w"),
    "KROATIE":            ("nos_kroatie", "KROATIË: Vluchtelingen krijgen stokslagen en schoppen van de politie", "NOS", "https://www.youtube.com/watch?v=YM9G3LDayBE"),
    "Moment_Calais":      ("bbc_calais", "Moment Calais migrants jumped onboard lorries", "BBC News", "https://www.youtube.com/watch?v=jvWlNsnM7eE"),
    "Waar_kunnen":        ("nosop3_opvang", "Waar kunnen de asielzoekers heen?", "NOS op 3", "https://www.youtube.com/watch?v=ONhbxpJp89I"),
    "Woede_over":         ("nieuwsuur_woede", "Woede over de vluchtelingenkwestie", "Nieuwsuur", "https://www.youtube.com/watch?v=K3r4JxQvVQk"),
    "YTDown.com":         ("ind_keten", "Wie doet wat in de vreemdelingenketen? Een film van IND, COA en DT&V", "Immigratie- en Naturalisatiedienst (IND)", "https://www.youtube.com/watch?v=xtm3LXngnSw"),
    "Zo_dodelijk":        ("nosop3_grenzen", "Zo dodelijk zijn Europa's grenzen", "NOS op 3", "https://www.youtube.com/watch?v=CYhe0bn5mnQ"),
}

assert not CSV_PATH.exists(), f"{CSV_PATH} already exists; the rename was already done"

files = sorted(p for p in VIDEO_DIR.iterdir() if p.suffix.lower() == ".mp4")
assert files, f"No .mp4 files in {VIDEO_DIR}"

# ---- Build the plan ----
counter, rows = Counter(), []
for src in files:
    matches = [k for k in SOURCES if src.name.startswith(k)]
    assert len(matches) == 1, f"No unique source for {src.name}"
    source_id, source_title, outlet, source_url = SOURCES[matches[0]]
    counter[source_id] += 1
    rows.append({
        "video_id": f"{source_id}_{counter[source_id]:02d}",
        "description": "",
        "source_id": source_id,
        "outlet": outlet,
        "source_title": source_title,
        "source_url": source_url,
        "original_filename": src.name,
    })

videos = pd.DataFrame(rows)
assert videos["video_id"].is_unique

print(videos[["video_id", "original_filename"]].to_string(index=False))
print(f"\n{len(videos)} clips from {videos['source_id'].nunique()} sources")
print(videos["source_id"].value_counts().sort_index())

# ---- Rename and write the CSV ----
if DRY_RUN:
    print("\nDRY RUN: nothing was changed. Set DRY_RUN = False to apply.")
else:
    for r in rows:
        (VIDEO_DIR / r["original_filename"]).rename(VIDEO_DIR / f"{r['video_id']}.mp4")
    videos.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"\nRenamed {len(rows)} clips and wrote {CSV_PATH}")

               video_id                                                                                                                                                 original_filename
   nieuwsuur_denhaag_01                                                                                                    De_problemen_met_arbeidsmigranten_in_Haagse_wijken00003230.mp4
   nieuwsuur_denhaag_02                                                                                                    De_problemen_met_arbeidsmigranten_in_Haagse_wijken00006013.mp4
   nieuwsuur_denhaag_03                                                                                                    De_problemen_met_arbeidsmigranten_in_Haagse_wijken00006719.mp4
   nieuwsuur_denhaag_04                                                                                                    De_problemen_met_arbeidsmigranten_in_Haagse_wijken00009506.mp4
   nieuwsuur_denhaag_05                                               